# Modelo de recomendación de cultivos con datos reales de Europa

Sustituye a `entrenar_modelo.ipynb`. Aquel dataset de Kaggle no servía para recomendar: en él el cultivo no depende de las condiciones (clasificarlo con todas sus variables acierta lo mismo que el azar), así que el modelo ordenaba siempre por t/ha y salía Sugarcane.

**Datos (todos reales, se descargan aquí, sin claves):**
- **Eurostat `apro_cpshr`**: superficie, producción y rendimiento (t/ha) de cada cultivo por región NUTS2 y año, 2010–2024.
- **NASA POWER**: clima mensual de cada región y año (temperatura, lluvia, radiación, humedad relativa).
- **GISCO**: punto interior de cada región, para pedir su clima.

**Dos modelos** (gradient boosting, con el cultivo y nueve variables de clima):
1. `se_cultiva`: probabilidad de que el cultivo se dé con ese clima.
2. `rendimiento`: t/ha esperadas.

El backend calcula las mismas nueve variables con las balizas y Meteomatics y ordena por `probabilidad × rendimiento × precio` (precio de venta de Eurostat).

Ejecuta todas las celdas (Entorno de ejecución → Ejecutar todas). La descarga tarda unos minutos la primera vez.

In [ ]:
import json, os, time, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict

print("scikit-learn", sklearn.__version__)

### 1. Descarga y construcción del conjunto de datos
Cada fila es un cultivo en una región y un año: si se cultiva (al menos 500 ha), su rendimiento y el clima de ese año. Las descargas se guardan en `cache/` para no repetirlas.

In [ ]:
EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
GISCO = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_LB_{}_4326_LEVL_2.geojson"
POWER = "https://power.larc.nasa.gov/api/temporal/monthly/point"
PRIMER_ANIO, ULTIMO_ANIO = 2010, 2024
CACHE = "cache"

# código de cultivo de Eurostat (apro_cpshr) -> (id en la tabla `cultivos`, nombre)
CULTIVOS = {
    "C1110": (1, "Trigo blando de invierno"), "C1120": (2, "Trigo duro"), "C1300": (3, "Cebada"),
    "C1410": (4, "Avena"), "C1500": (5, "Maíz grano"), "C1700": (7, "Sorgo grano"), "C2000": (8, "Arroz"),
    "C1200": (58, "Centeno"), "C1600": (59, "Triticale"), "R1000": (20, "Patata"),
    "R2000": (19, "Remolacha azucarera"), "I1110": (16, "Colza"), "I1120": (15, "Girasol"),
    "I1130": (65, "Soja"), "I3000": (68, "Tabaco"), "P1200": (11, "Haba seca"),
    "I1140": (67, "Lino"), "G3000": (61, "Maíz forrajero (ensilado)"),
}
AREA_MINIMA_KHA = 0.5      # un cultivo "se da" en una región-año si ocupa al menos 500 ha
MIN_CULTIVOS_REGION = 5    # región-año que informa de al menos estos cultivos (para los ceros)


def _json(url, intentos=4):
    for i in range(intentos):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                return json.loads(r.read().decode())
        except Exception:
            if i == intentos - 1:
                raise
            time.sleep(3 * (i + 1))


def _en_cache(nombre, fn):
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, nombre)
    if os.path.exists(ruta):
        return json.load(open(ruta))
    datos = fn()
    json.dump(datos, open(ruta, "w"))
    return datos


def _filas(d):
    ids, sizes = d["id"], d["size"]
    codes = [sorted(d["dimension"][k]["category"]["index"], key=d["dimension"][k]["category"]["index"].get) for k in ids]
    for k, v in d["value"].items():
        k, pos = int(k), []
        for s in reversed(sizes):
            k, i = divmod(k, s)
            pos.append(i)
        pos.reverse()
        yield {dim: codes[j][i] for j, (dim, i) in enumerate(zip(ids, pos))}, v


def rendimientos_regionales():
    """Eurostat apro_cpshr -> DataFrame región, año, cultivo, area_kha, rendimiento_t_ha (None si no hay)."""
    bruto = {}
    cultivos = "&".join(f"crops={c}" for c in CULTIVOS)
    for medida in ("YLD_HUMD_EU_T_HA", "AR_THS_HA", "HPRD_HUMD_EU_THS_T"):
        url = f"{EUROSTAT}/apro_cpshr?format=JSON&lang=EN&strucpro={medida}&{cultivos}&sinceTimePeriod={PRIMER_ANIO}&untilTimePeriod={ULTIMO_ANIO}"
        d = _en_cache(f"cpshr_{medida}.json", lambda: _json(url))
        for fila, valor in _filas(d):
            if len(fila["geo"]) == 4 and not fila["geo"].endswith("ZZ"):   # solo regiones NUTS2
                bruto.setdefault((fila["geo"], int(fila["time"]), fila["crops"]), {})[medida] = valor
    filas = []
    for (region, anio, cultivo), v in bruto.items():
        area, prod = v.get("AR_THS_HA"), v.get("HPRD_HUMD_EU_THS_T")
        rend = v.get("YLD_HUMD_EU_T_HA") or (prod / area if area and prod else None)
        filas.append({"region": region, "anio": anio, "cultivo": cultivo, "area_kha": area or 0.0,
                      "rendimiento_t_ha": rend if rend and rend > 0 else None})
    return pd.DataFrame(filas)


def centroides():
    """Punto interior de cada región NUTS2 (GISCO; varias versiones de NUTS porque los códigos cambian)."""
    puntos = {}
    for version in (2016, 2021, 2024):   # la más nueva pisa a las anteriores
        d = _en_cache(f"nuts_{version}.json", lambda: _json(GISCO.format(version)))
        for f in d["features"]:
            lon, lat = f["geometry"]["coordinates"]
            puntos[f["properties"]["NUTS_ID"]] = (round(lat, 3), round(lon, 3))
    return puntos


def clima_mensual(lat, lon):
    """NASA POWER: {(año, mes): {t, lluvia_mm_dia, rad_mj, hr}} del punto."""
    q = urllib.parse.urlencode({"parameters": "T2M,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M", "community": "AG",
                                "longitude": lon, "latitude": lat, "start": PRIMER_ANIO, "end": ULTIMO_ANIO, "format": "JSON"})
    p = _en_cache(f"power_{lat}_{lon}.json", lambda: _json(f"{POWER}?{q}")["properties"]["parameter"])
    claves = {"T2M": "t", "PRECTOTCORR": "lluvia_mm_dia", "ALLSKY_SFC_SW_DWN": "rad_mj", "RH2M": "hr"}
    meses = {}
    for param, clave in claves.items():
        for fecha, valor in p[param].items():
            anio, mes = int(fecha[:4]), int(fecha[4:])
            if 1 <= mes <= 12 and valor is not None and valor > -900:
                meses.setdefault((anio, mes), {})[clave] = valor
    return meses


DIAS_MES = [31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
VARIABLES = ["t_anual", "t_mes_frio", "t_mes_calido", "t_primavera", "lluvia_anual_mm", "lluvia_abr_sep_mm",
             "radiacion_mj_dia", "radiacion_abr_sep_mj_dia", "hr_media"]


def variables_del_anio(meses):
    """Las variables del modelo a partir de 12 meses {mes: {t, lluvia_mm_dia, rad_mj, hr}}. None si falta algo.
    El backend calcula las mismas con los datos de las balizas y Meteomatics."""
    if any(m not in meses or len(meses[m]) < 4 for m in range(1, 13)):
        return None
    t = [meses[m]["t"] for m in range(1, 13)]
    lluvia = [meses[m]["lluvia_mm_dia"] * DIAS_MES[m - 1] for m in range(1, 13)]
    rad = [meses[m]["rad_mj"] for m in range(1, 13)]
    return {
        "t_anual": sum(t) / 12, "t_mes_frio": min(t), "t_mes_calido": max(t), "t_primavera": sum(t[2:6]) / 4,
        "lluvia_anual_mm": sum(lluvia), "lluvia_abr_sep_mm": sum(lluvia[3:9]),
        "radiacion_mj_dia": sum(rad) / 12, "radiacion_abr_sep_mj_dia": sum(rad[3:9]) / 6,
        "hr_media": sum(meses[m]["hr"] for m in range(1, 13)) / 12,
    }


def construir():
    rend = rendimientos_regionales()
    puntos = centroides()
    regiones = sorted(set(rend.region) & set(puntos))
    print(f"regiones con datos: {rend.region.nunique()}, con centroide: {len(regiones)}")
    with ThreadPoolExecutor(4) as pool:
        climas = dict(zip(regiones, pool.map(lambda r: clima_mensual(*puntos[r]), regiones)))
    filas = []
    for region in regiones:
        for anio in range(PRIMER_ANIO, ULTIMO_ANIO + 1):
            v = variables_del_anio({m: climas[region].get((anio, m), {}) for m in range(1, 13)})
            if v:
                filas.append({"region": region, "anio": anio, "latitud": puntos[region][0], "longitud": puntos[region][1], **v})
    clima = pd.DataFrame(filas)

    # Presencia: en cada región-año que informa de bastantes cultivos, cada cultivo de la lista se da (1) o no (0).
    informa = rend[rend.area_kha > 0].groupby(["region", "anio"]).cultivo.nunique()
    validas = informa[informa >= MIN_CULTIVOS_REGION].reset_index()[["region", "anio"]]
    rejilla = validas.merge(pd.DataFrame({"cultivo": list(CULTIVOS)}), how="cross")
    datos = rejilla.merge(rend, on=["region", "anio", "cultivo"], how="left")
    datos["area_kha"] = datos.area_kha.fillna(0.0)
    datos["se_cultiva"] = (datos.area_kha >= AREA_MINIMA_KHA).astype(int)
    # Rendimientos imposibles (errores de unidades en origen): fuera los que pasan de 4 veces la mediana del cultivo.
    mediana = datos.groupby("cultivo").rendimiento_t_ha.transform("median")
    datos.loc[(datos.rendimiento_t_ha > 4 * mediana) | (datos.se_cultiva == 0), "rendimiento_t_ha"] = None
    datos = datos.merge(clima, on=["region", "anio"])
    datos["cultivo_id"] = datos.cultivo.map(lambda c: CULTIVOS[c][0])
    datos["cultivo_nombre"] = datos.cultivo.map(lambda c: CULTIVOS[c][1])
    return datos

In [ ]:
datos = construir()
datos.to_csv("entrenamiento_eurostat_power.csv.gz", index=False, float_format="%.4g",
             columns=[c for c in datos.columns if c not in ("cultivo_nombre", "cultivo_id")])
print(datos.shape, "| regiones:", datos.region.nunique(), "| países:", datos.region.str[:2].nunique(),
      "| años:", datos.anio.min(), "-", datos.anio.max())
print("filas con rendimiento:", datos.rendimiento_t_ha.notna().sum())
print(datos.groupby("cultivo_nombre").agg(filas_con_rendimiento=("rendimiento_t_ha", "count"), se_cultiva=("se_cultiva", "mean"),
                                         rendimiento_mediano=("rendimiento_t_ha", "median")).round(2).to_string())

### 2. Evaluación
Validación cruzada dejando **regiones enteras fuera**: mide si el modelo acierta en sitios que no ha visto, que es como se usa (una finca nueva). El rendimiento se predice relativo a la mediana de cada cultivo para que la patata (30 t/ha) no pese más que el trigo (5 t/ha); un R² de 0 sería decir siempre la mediana.

In [ ]:
codigos = sorted(CULTIVOS)                      # la posición es el código numérico del cultivo
datos["cultivo_cod"] = datos.cultivo.map(codigos.index)
COLUMNAS = ["cultivo_cod"] + VARIABLES
cv = GroupKFold(n_splits=5)

def clasificador():
    return HistGradientBoostingClassifier(categorical_features=[0], max_iter=300, learning_rate=0.08, random_state=0)

def regresor():
    return HistGradientBoostingRegressor(categorical_features=[0], max_iter=400, learning_rate=0.06, random_state=0)

# ¿Se cultiva?
X, y = datos[COLUMNAS], datos.se_cultiva
p = cross_val_predict(clasificador(), X, y, cv=cv, groups=datos.region, method="predict_proba")[:, 1]
auc = roc_auc_score(y, p)
print(f"SE CULTIVA: AUC {auc:.3f} en regiones no vistas (0,5 = azar)")

# Rendimiento
r = datos[datos.rendimiento_t_ha.notna()].copy()
mediana = r.groupby("cultivo").rendimiento_t_ha.median()
r["rel"] = r.rendimiento_t_ha / r.cultivo.map(mediana)
pr = cross_val_predict(regresor(), r[COLUMNAS], r.rel, cv=cv, groups=r.region)
r["pred"] = pr * r.cultivo.map(mediana)
r2 = r2_score(r.rel, pr)
print(f"RENDIMIENTO: R² {r2:.3f} en regiones no vistas (0 = decir siempre la mediana del cultivo)")

filas = []
for c in codigos:
    g, sel = r[r.cultivo == c], datos.cultivo == c
    filas.append({"cultivo": CULTIVOS[c][1], "AUC se cultiva": round(roc_auc_score(y[sel], p[sel]), 2), "filas": len(g),
                  "mediana t/ha": round(mediana[c], 2), "R² rendimiento": round(r2_score(g.rendimiento_t_ha, g.pred), 2),
                  "error medio t/ha": round(mean_absolute_error(g.rendimiento_t_ha, g.pred), 2),
                  "error diciendo la mediana": round(mean_absolute_error(g.rendimiento_t_ha, np.full(len(g), mediana[c])), 2)})
print(pd.DataFrame(filas).to_string(index=False))

### 3. Modelo final y exportación
Se entrena con todos los datos y se guarda en un solo archivo. Cópialo, junto con el conjunto de datos, al backend:

- `modelo_cultivos_europa.joblib` → `modelo_predicciones/`
- `entrenamiento_eurostat_power.csv.gz` → `modelo_predicciones/datos/`

Si el backend tiene otra versión de scikit-learn y el `.joblib` no carga, el backend entrena solo al arrancar con ese mismo conjunto de datos y estos mismos parámetros (`modelo_predicciones/modelo_europa.py`).

In [ ]:
m_cultiva = clasificador().fit(X, y)
m_rendimiento = regresor().fit(r[COLUMNAS], r.rel)
meta = {
    "columnas": COLUMNAS, "variables": VARIABLES, "codigos": codigos,
    "cultivos": {c: {"id": CULTIVOS[c][0], "nombre": CULTIVOS[c][1], "mediana_t_ha": float(mediana[c])} for c in codigos},
    "rangos_entrenamiento": {v: [float(datos[v].quantile(0.01)), float(datos[v].quantile(0.99))] for v in VARIABLES},
    "sklearn": sklearn.__version__,
    "evaluacion": {"auc_se_cultiva": float(auc), "r2_rendimiento_relativo": float(r2)},
    "datos": "Eurostat apro_cpshr 2010-2024 (NUTS2) + NASA POWER mensual",
}
joblib.dump({"se_cultiva": m_cultiva, "rendimiento": m_rendimiento, "meta": meta}, "modelo_cultivos_europa.joblib", compress=3)
print("guardado modelo_cultivos_europa.joblib,", os.path.getsize("modelo_cultivos_europa.joblib") // 1024, "KB")

try:
    from google.colab import files
    files.download("modelo_cultivos_europa.joblib")
    files.download("entrenamiento_eurostat_power.csv.gz")
except ImportError:
    pass  # fuera de Colab: los archivos quedan en esta carpeta

### 4. Prueba: qué recomienda con un clima dado
Ejemplo con el clima del último año de la finca de Wexford (Irlanda). El backend añade el precio de venta para ordenar por ingreso.

In [ ]:
def recomendar(variables):
    marco = pd.DataFrame([[i] + [variables[v] for v in VARIABLES] for i in range(len(codigos))], columns=COLUMNAS)
    prob = m_cultiva.predict_proba(marco)[:, 1]
    rend = m_rendimiento.predict(marco)
    tabla = pd.DataFrame({"cultivo": [CULTIVOS[c][1] for c in codigos], "probabilidad %": (100 * prob).round(1),
                          "rendimiento t/ha": [max(0, x) * mediana[c] for x, c in zip(rend, codigos)]})
    return tabla.sort_values("probabilidad %", ascending=False).round(2)

print(recomendar({"t_anual": 11.2, "t_mes_frio": 5.7, "t_mes_calido": 17.8, "t_primavera": 11.6, "lluvia_anual_mm": 1016,
                  "lluvia_abr_sep_mm": 370, "radiacion_mj_dia": 10.2, "radiacion_abr_sep_mj_dia": 16.3, "hr_media": 82.1}).to_string(index=False))